# Delta Hedging ComparisonThis notebook generates synthetic underlying paths, computes hedge ratios from the Black–Scholes model and the binomial pricers, and compares the hedging performance through rebalancing simulations.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from Binomial_Pricing import Cox_Ross_Rubinstein_Tree, Jarrow_Rudd_Tree

np.random.seed(42)
plt.style.use('seaborn-v0_8')


In [ ]:
# Scenario parameters
S0 = 100.0
K = 100.0
r = 0.02
sigma = 0.25
T = 1.0
steps = 50
n_paths = 2000
option_type = 'C'

dt = T / steps


In [ ]:
# Helper utilities

def norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def black_scholes_price_delta(S, K, r, sigma, tau, option_type='C'):
    if tau <= 0:
        payoff = max(S - K, 0.0) if option_type == 'C' else max(K - S, 0.0)
        intrinsic_delta = 1.0 if (option_type == 'C' and S > K) else -1.0 if (option_type == 'P' and S < K) else 0.0
        return payoff, intrinsic_delta

    vol_term = sigma * math.sqrt(tau)
    d1 = (math.log(S / K) + (r + 0.5 * sigma ** 2) * tau) / vol_term
    d2 = d1 - vol_term

    if option_type == 'C':
        price = S * norm_cdf(d1) - K * math.exp(-r * tau) * norm_cdf(d2)
        delta = norm_cdf(d1)
    else:
        price = K * math.exp(-r * tau) * norm_cdf(-d2) - S * norm_cdf(-d1)
        delta = norm_cdf(d1) - 1.0

    return price, delta


def crr_price_delta(S, K, r, sigma, tau, remaining_steps, option_type='C'):
    if tau <= 0 or remaining_steps <= 0:
        payoff = max(S - K, 0.0) if option_type == 'C' else max(K - S, 0.0)
        intrinsic_delta = 1.0 if (option_type == 'C' and S > K) else -1.0 if (option_type == 'P' and S < K) else 0.0
        return payoff, intrinsic_delta

    dt_local = tau / remaining_steps
    u = math.exp(sigma * math.sqrt(dt_local))
    d = math.exp(-sigma * math.sqrt(dt_local))
    pu = (math.exp(r * dt_local) - d) / (u - d)
    disc = math.exp(-r * dt_local)

    if remaining_steps == 1:
        payoff_up = max(S * u - K, 0.0) if option_type == 'C' else max(K - S * u, 0.0)
        payoff_down = max(S * d - K, 0.0) if option_type == 'C' else max(K - S * d, 0.0)
        price = disc * (pu * payoff_up + (1 - pu) * payoff_down)
        delta = (payoff_up - payoff_down) / (S * u - S * d)
        return price, delta

    price_up = Cox_Ross_Rubinstein_Tree(S * u, K, tau - dt_local, r, sigma, remaining_steps - 1, option_type)
    price_down = Cox_Ross_Rubinstein_Tree(S * d, K, tau - dt_local, r, sigma, remaining_steps - 1, option_type)

    price = disc * (pu * price_up + (1 - pu) * price_down)
    delta = (price_up - price_down) / (S * u - S * d)
    return price, delta


def generate_gbm_paths(S0, r, sigma, T, steps, n_paths):
    dt = T / steps
    increments = (r - 0.5 * sigma ** 2) * dt + sigma * math.sqrt(dt) * np.random.randn(n_paths, steps)
    log_paths = np.cumsum(increments, axis=1)
    log_paths = np.hstack([np.zeros((n_paths, 1)), log_paths])
    paths = S0 * np.exp(log_paths)
    return paths


In [ ]:
# Hedging simulation

def simulate_delta_hedge(paths, pricer='bs', transaction_cost_rate=0.0):
    n_steps = paths.shape[1] - 1
    dt = T / n_steps
    pnl = []

    for path in paths:
        tau = T
        if pricer == 'bs':
            option_price, delta = black_scholes_price_delta(path[0], K, r, sigma, tau, option_type)
        else:
            option_price, delta = crr_price_delta(path[0], K, r, sigma, tau, n_steps, option_type)

        cash = option_price - delta * path[0]

        for t in range(1, len(path)):
            tau = T - (t - 1) * dt
            S_prev = path[t - 1]

            if pricer == 'bs':
                _, new_delta = black_scholes_price_delta(S_prev, K, r, sigma, tau, option_type)
            else:
                remaining_steps = n_steps - (t - 1)
                _, new_delta = crr_price_delta(S_prev, K, r, sigma, tau, remaining_steps, option_type)

            delta_change = new_delta - delta
            trade_cost = delta_change * S_prev
            fee = abs(trade_cost) * transaction_cost_rate
            cash = cash * math.exp(r * dt) - trade_cost - fee
            delta = new_delta

        cash = cash * math.exp(r * dt)
        payoff = max(path[-1] - K, 0.0) if option_type == 'C' else max(K - path[-1], 0.0)
        portfolio = cash + delta * path[-1]
        pnl.append(portfolio - payoff)

    return np.array(pnl)


paths = generate_gbm_paths(S0, r, sigma, T, steps, n_paths)
bs_pnl = simulate_delta_hedge(paths, pricer='bs', transaction_cost_rate=0.0005)
binomial_pnl = simulate_delta_hedge(paths, pricer='binomial', transaction_cost_rate=0.0005)

summary = pd.DataFrame({
    'model': ['Black-Scholes', 'Binomial'],
    'mean_pnl': [bs_pnl.mean(), binomial_pnl.mean()],
    'std_pnl': [bs_pnl.std(), binomial_pnl.std()],
    'median_pnl': [np.median(bs_pnl), np.median(binomial_pnl)]
})
summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
axes[0].hist(bs_pnl, bins=40, color='tab:blue', alpha=0.75)
axes[0].set_title('Black-Scholes hedge P&L')
axes[0].set_xlabel('P&L')
axes[0].set_ylabel('Frequency')

axes[1].hist(binomial_pnl, bins=40, color='tab:orange', alpha=0.75)
axes[1].set_title('Binomial hedge P&L')
axes[1].set_xlabel('P&L')

fig.suptitle('Delta hedge error distributions')
plt.tight_layout()
plt.show()
